# Nasdaq Closing Auction Forecasting Research

**Sohan Hajra**

Research framework for predicting final Nasdaq closing-cross displacement from information available during the pre-close auction.

> **Data note:** The source dataset used in the original research is not redistributed in this public portfolio version. Execution outputs have been cleared and the notebook expects a local archive at `data/nasdaq_closing_auction.tar`.


## 1. Setup and observation

I first loaded one trading day to understand the structure of the Nasdaq closing-auction data before doing any modeling.

I used AAPL as a liquid example and looked at how the bid/ask quotes, paired shares, imbalance shares, side, Reference price, Near/Far prices, and final closing cross change as 4:00 PM gets closer. The goal here is mainly to understand what each field represents and when the auction information becomes available.

In [ ]:
import tarfile
import gzip
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

In [ ]:
# Open one sample trading day and inspect the schema.
tar_path = "data/nasdaq_closing_auction.tar"

with tarfile.open(tar_path, "r") as tar:
    member_names = sorted(
        m.name
        for m in tar.getmembers()
        if m.name.endswith(".csv.gz")
    )
    member_name = member_names[0]
    member = tar.getmember(member_name)
    raw_file = tar.extractfile(member)

    with gzip.GzipFile(fileobj=raw_file) as gz:
        df = pd.read_csv(gz)

print(df.shape)
print(df.columns.tolist())


In [ ]:
# Picking AAPL to observe what the numbers mean and get an idea of how the rows are filled in the table

aapl = df[df["symbol"] == "AAPL"].copy()

print(aapl.shape)

print(
    aapl[
        [
            "local_time",
            "bid",
            "ask",
            "paired_shares",
            "shares",
            "side",
            "ref_price",
            "near_price",
            "far_price",
            "cross",
        ]
    ].head(20).to_string(index=False)
)

In [ ]:
# Checking the changes in regime from 15:55 onwards and checking what happens at 16:00 pm

cols = [
    "local_time",
    "bid",
    "ask",
    "paired_shares",
    "shares",
    "side",
    "ref_price",
    "near_price",
    "far_price",
    "cross"
]

print(aapl[cols].iloc[25:45].to_string(index=False))

print(aapl[cols].tail(25).to_string(index=False))

print("AAPL non-null cross values:", aapl["cross"].notna().sum())

print(aapl.loc[aapl["cross"].notna(), cols].to_string(index=False))

print("Total non-null cross values:", df["cross"].notna().sum())

In [ ]:
#checking if there is only one unique cross value or if it changes or modified after 4:00 pm for any of the stocks

cross_counts = (
    df.groupby("symbol")["cross"]
      .nunique(dropna=True)
)

print(cross_counts.value_counts().sort_index())

print("Symbols with more than one unique cross:")
print(cross_counts[cross_counts > 1])

In [ ]:
# how many symbols we are dealing with
print("Number of symbols:", df["symbol"].nunique())

print("First 20 symbols:")
print(sorted(df["symbol"].dropna().unique())[:20])

In [ ]:
#inspecting where AAPL happens to cross from pre close to post close

print(
    aapl.loc[
        aapl["local_time"].str.contains(
            r"15:59:5|16:00:0",
            regex=True
        ),
        cols
    ].to_string(index=False)
)

## 2. Construct pre-close observations and closing-cross targets

The dataset contains auction messages from both before and after 4:00 PM. For the prediction problem, I only used information that was available before the close.

The post-close records are only used to identify the actual closing-cross price for each stock-day. This keeps the predictors separated from the outcome and avoids using information that would not have been known at prediction time.

In [ ]:
df["local_time"] = pd.to_datetime(df["local_time"])

df["date"] = df["local_time"].dt.date

In [ ]:
#Seperating pre close rows and constructing the target table
pre_close = df[
    df["local_time"].dt.time < pd.Timestamp("16:00:00").time()
].copy()

targets = (
    df.dropna(subset=["cross"])
      .groupby(["date", "symbol"], as_index=False)
      .agg(cross=("cross", "first"))
)

print("Pre-close rows:", len(pre_close))
print("Target rows:", len(targets))
print()
print(targets.head(10).to_string(index=False))

print()
print(targets[targets["symbol"] == "AAPL"].to_string(index=False))

## 3. Construct the prediction dataset

For each pre-close auction message, I attached the final closing-cross price for the same date and symbol. This gave me a prediction target for every point in the pre-close window.

I defined the target as the closing-cross move relative to the bid-ask midpoint at that time, measured in basis points:

$$
y_t =
10{,}000
\left(
\frac{C}{M_t}-1
\right),
\qquad
M_t = \frac{\text{bid}_t+\text{ask}_t}{2}.
$$

Using basis points makes the target and prediction errors easier to compare across stocks with very different price levels.

In [ ]:
#creating the supervised learning dataset
targets = (
    df.dropna(subset=["cross"])
      .groupby(["date", "symbol"], as_index=False)
      .agg(final_cross=("cross", "first"))
)

pre_close = pre_close.drop(columns=["cross"])

model_df = pre_close.merge(
    targets,
    on=["date", "symbol"],
    how="inner",
    validate="many_to_one"
)

print("Pre-close rows:", len(pre_close))
print("Merged rows:", len(model_df))
print("Missing final cross:", model_df["final_cross"].isna().sum())

In [ ]:
#constructing concurrent midpoint and check for invalid bbo
invalid_bbo = (
    model_df["bid"].isna()
    | model_df["ask"].isna()
    | (model_df["bid"] <= 0)
    | (model_df["ask"] <= 0)
    | (model_df["ask"] < model_df["bid"])
)

print("Invalid BBO rows:", invalid_bbo.sum())


model_df["mid"] = (
    model_df["bid"] + model_df["ask"]
) / 2

In [ ]:
#constructing dependent variable y_t
model_df["cross_move_bps"] = 10_000 * (
    model_df["final_cross"] / model_df["mid"] - 1
)
aapl_model = model_df[
    model_df["symbol"] == "AAPL"
].copy()

display_cols = [
    "local_time",
    "bid",
    "ask",
    "mid",
    "near_price",
    "far_price",
    "final_cross",
    "cross_move_bps"
]

print(aapl_model[display_cols].head(5).to_string(index=False))

print()

print(aapl_model[display_cols].tail(10).to_string(index=False))

print()

print(
    model_df["cross_move_bps"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

In [ ]:
#Checking the invalid BBO rows
print(
    model_df.loc[
        invalid_bbo,
        [
            "date",
            "symbol",
            "local_time",
            "bid",
            "ask",
            "final_cross",
            "mid",
            "cross_move_bps"
        ]
    ].to_string(index=False))
print()

#checking the extreme negative observations
extreme_negative = model_df[
    model_df["cross_move_bps"] <= -1000
]

print("Extreme negative rows:", len(extreme_negative))
print()

print(
    extreme_negative[
        [
            "date",
            "symbol",
            "local_time",
            "bid",
            "ask",
            "mid",
            "final_cross",
            "cross_move_bps"
        ]
    ].head(30).to_string(index=False)
)
print()

print(
    targets["final_cross"]
    .sort_values()
    .head(20)
    .to_string(index=False)
)
print()

print(
    targets[
        targets["final_cross"] <= 0
    ].to_string(index=False)
)
print()

#checking how many stock days cause these extreme negative observations
print(
    extreme_negative[
        ["date", "symbol", "final_cross"]
    ]
    .drop_duplicates()
    .to_string(index=False)
)
print()

#checking what the distribution would look like if we remove the extremes
reasonable = model_df[
    model_df["cross_move_bps"].between(-500, 500)
]

print(
    reasonable["cross_move_bps"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

### 3.1 Data quality filtering

Before modeling, I audit target validity at the stock-day level and quote validity at the individual snapshot level.

Stock-days with structurally invalid closing-cross targets are removed from the prediction sample. Individual snapshots with unavailable or invalid bid/ask quotes are removed only when a midpoint cannot be constructed.

I do not clip or winsorize the target. Large observations are inspected first and retained unless there is a clear structural data problem.


In [ ]:
#filtering the data to remove the invalid rows that we identified
valid_target = model_df["final_cross"] > 0

valid_bbo = (
    model_df["bid"].notna()
    & model_df["ask"].notna()
    & (model_df["bid"] > 0)
    & (model_df["ask"] > 0)
    & (model_df["ask"] >= model_df["bid"])
)

clean_df = model_df[
    valid_target & valid_bbo
].copy()

#recomputing the midpoint and target after filtering
clean_df["mid"] = (
    clean_df["bid"] + clean_df["ask"]
) / 2

clean_df["cross_move_bps"] = 10_000 * (
    clean_df["final_cross"] / clean_df["mid"] - 1
)


print("Rows before cleaning:", len(model_df))
print()

print("Rows after cleaning:", len(clean_df))
print()

print("Rows removed:", len(model_df) - len(clean_df))
print()

print(
    "Remaining stock-day targets:",
    clean_df[["date", "symbol"]].drop_duplicates().shape[0]
)
print()

print("Missing midpoint:", clean_df["mid"].isna().sum())
print()

print(
    "Missing target:",
    clean_df["cross_move_bps"].isna().sum()
)
print()

print(
    clean_df["cross_move_bps"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

## 4. Load and validate the full dataset

After validating the target-construction and cleaning logic on a single trading day, I apply the same process across the full archive.

The full-sample audit is designed to verify that each stock-day has a unique final closing-cross target, that pre-close observations are consistently sampled, and that invalid quotes are handled at the snapshot level rather than by dropping an entire stock-day unnecessarily.

The midpoint-relative closing-cross displacement is then recomputed on the clean sample and inspected both overall and by trading date.


In [ ]:
#load files
with tarfile.open(tar_path, "r") as tar:
    daily_files = sorted(
        member.name
        for member in tar.getmembers()
        if member.name.endswith(".csv.gz")
    )

print("Number of daily files:", len(daily_files))
print(daily_files)

In [ ]:
#read every daily file and create the full dataset
daily_dfs = []

with tarfile.open(tar_path, "r") as tar:
    for member_name in daily_files:
        member = tar.getmember(member_name)
        raw_file = tar.extractfile(member)

        with gzip.GzipFile(fileobj=raw_file) as gz:
            daily_df = pd.read_csv(gz)

        daily_df["source_file"] = member_name
        daily_dfs.append(daily_df)

full_df = pd.concat(
    daily_dfs,
    ignore_index=True
)

In [ ]:
print("Full dataset shape:", full_df.shape)
print()
print("Number of symbols:", full_df["symbol"].nunique())
print()
print("Number of source files:", full_df["source_file"].nunique())
print()

print(
    full_df["source_file"]
    .value_counts()
    .sort_index()
)

In [ ]:
#handling the timestamps
full_df["local_time"] = (
    pd.to_datetime(
        full_df["local_time"],
        utc=True
    )
    .dt.tz_convert("America/New_York")
)

full_df["date"] = full_df["local_time"].dt.date

#checking if the ts colummns are the same or not
same_ts = (
    full_df["ts"].astype(str)
    ==
    full_df["ts.1"].astype(str)
)

print("Duplicate timestamps identical:", same_ts.all())
print("Rows where timestamps differ:", (~same_ts).sum())

full_df = full_df.drop(columns=["ts.1"])

In [ ]:
#checking stock day message counts
stock_day_counts = (
    full_df
    .groupby(["date", "symbol"])
    .size()
)

print(
    stock_day_counts
    .describe()
)
print()

print(
    stock_day_counts
    .value_counts()
    .sort_index()
)
print()

#number of symbols
symbols_per_day = (
    full_df
    .groupby("date")["symbol"]
    .nunique()
)

print(symbols_per_day)
print()

#unique stock days to check whether there are 21*500 = 10.5k stock days
n_stock_days = (
    full_df[
        ["date", "symbol"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Total stock-days:", n_stock_days)

In [ ]:
#checking the interesting number of messages for some of the stock days
#checking pre close messages per stock day
pre_close_full = full_df[
    full_df["local_time"].dt.time
    < pd.Timestamp("16:00:00").time()
].copy()

pre_close_counts = (
    pre_close_full
    .groupby(["date", "symbol"])
    .size()
)

print(pre_close_counts.describe())
print()

print(
    pre_close_counts
    .value_counts()
    .sort_index()
)
print()

#abnormal pre close stock days
abnormal_pre_counts = pre_close_counts[
    pre_close_counts != 330
]

print(
    abnormal_pre_counts
    .sort_values()
)
print()

#checking the large raw count cases
large_stock_days = (
    stock_day_counts[stock_day_counts > 500]
    .rename("count")
    .reset_index()
)

print(large_stock_days)

for row in large_stock_days.itertuples(index=False):
    date = row.date
    symbol = row.symbol
    count = row.count

    print()
    print(date, symbol, count)

    temp = full_df[
        (full_df["date"] == date)
        & (full_df["symbol"] == symbol)
    ]

    print(
        temp["local_time"]
        .dt.floor("s")
        .value_counts()
        .sort_index()
        .head(20)
    )

In [ ]:
#checking every cross prices
target_audit = (
    full_df
    .groupby(["date", "symbol"])["cross"]
    .agg(
        cross_observations=lambda s: s.notna().sum(),
        unique_crosses=lambda s: s.dropna().nunique(),
        final_cross=lambda s: (
            s.dropna().iloc[0]
            if not s.dropna().empty
            else np.nan
        )
    )
    .reset_index()
)


print(
    target_audit["unique_crosses"]
    .value_counts()
    .sort_index()
)
print()

print(
    "Stock-days with no cross:",
    target_audit["final_cross"].isna().sum()
)
print()

print(
    "Stock-days with non-positive cross:",
    (target_audit["final_cross"] <= 0).sum()
)
print()

print(
    "Stock-days with conflicting crosses:",
    (target_audit["unique_crosses"] > 1).sum()
)
print()

print(
    target_audit["cross_observations"]
    .value_counts()
    .sort_index()
)

In [ ]:
# checking invalid stock-day targets
invalid_targets = target_audit[
    target_audit["final_cross"].isna()
    | (target_audit["final_cross"] <= 0)
    | (target_audit["unique_crosses"] != 1)
]

print(
    invalid_targets
    .sort_values(["date", "symbol"])
    .to_string(index=False)
)

In [ ]:
#building the valid target table
valid_targets_full = target_audit[
    (target_audit["unique_crosses"] == 1)
    & (target_audit["final_cross"] > 0)
][
    ["date", "symbol", "final_cross"]
].copy()

print("Valid target stock-days:", len(valid_targets_full))

In [ ]:
#removing the pre close cross column which shows NaN
pre_close_full = pre_close_full.drop(
    columns=["cross"],
    errors="ignore"
)

In [ ]:
full_model_df = pre_close_full.merge(
    valid_targets_full,
    on=["date", "symbol"],
    how="inner",
    validate="many_to_one"
)

print("Original pre-close rows:", len(pre_close_full))
print("Merged valid-target rows:", len(full_model_df))
print()

print(
    "Valid stock-days:",
    full_model_df[["date", "symbol"]]
    .drop_duplicates()
    .shape[0]
)
print()

print(
    "Missing final cross:",
    full_model_df["final_cross"].isna().sum()
)

In [ ]:
#checking for invalid bbo rows
invalid_bbo_full = (
    full_model_df["bid"].isna()
    | full_model_df["ask"].isna()
    | (full_model_df["bid"] <= 0)
    | (full_model_df["ask"] <= 0)
    | (full_model_df["ask"] < full_model_df["bid"])
)

print(
    "Invalid BBO rows:",
    invalid_bbo_full.sum()
)

print(
    "Missing bid:",
    full_model_df["bid"].isna().sum()
)

print(
    "Missing ask:",
    full_model_df["ask"].isna().sum()
)

print(
    "Non-positive bid:",
    (full_model_df["bid"] <= 0).sum()
)

print(
    "Non-positive ask:",
    (full_model_df["ask"] <= 0).sum()
)

print(
    "Crossed BBO:",
    (
        full_model_df["ask"]
        < full_model_df["bid"]
    ).sum()
)
print()

# where the invalid bbo rows are
bad_bbo_stock_days = (
    full_model_df.loc[
        invalid_bbo_full,
        ["date", "symbol"]
    ]
    .value_counts()
    .reset_index(name="bad_rows")
)

print(bad_bbo_stock_days.to_string(index=False))

In [ ]:
#creating the clean full month dataset
clean_full_df = full_model_df[
    ~invalid_bbo_full
].copy()

clean_full_df["mid"] = (clean_full_df["bid"]+ clean_full_df["ask"]) / 2

clean_full_df["cross_move_bps"] = 10_000 * (
    clean_full_df["final_cross"]
    / clean_full_df["mid"]
    - 1
)

print("Rows before BBO cleaning:", len(full_model_df))
print("Rows after BBO cleaning:", len(clean_full_df))
print(
    "Rows removed for invalid BBO:",
    len(full_model_df) - len(clean_full_df)
)

print(
    "Remaining stock-days:",
    clean_full_df[
        ["date", "symbol"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Missing midpoint:",
    clean_full_df["mid"].isna().sum()
)

print(
    "Missing target:",
    clean_full_df["cross_move_bps"].isna().sum()
)
print()

print(
    clean_full_df["cross_move_bps"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
#checking the metrics for the full dataset
daily_target_summary = (
    clean_full_df
    .groupby("date")["cross_move_bps"]
    .agg(["mean", "median", "std", "count"])
)

print(daily_target_summary)

## 5. Feature engineering and auction signal analysis

Before building a model, I examine the main auction variables individually to understand what information they may contain about the final cross.

Features are studied at fixed times before the close so that stocks are compared at the same stage of the auction. Near, Far, and Reference prices are expressed relative to the current midpoint. I also examine imbalance relative to ADV and auction volume, paired volume, spread, top-of-book imbalance, and the stock's move from the open.

For each feature, I first check missing values, sentinel values, and unusual observations before deciding how to treat them. I then use correlations, MAE comparisons, side-based analysis, and residual analysis to test whether a feature adds information beyond simpler market-price baselines.

The public version keeps the full feature-engineering and evaluation code while omitting the private dataset and dataset-specific outputs.


### 5.1 Indicative auction price signals: Near and Far prices

In [ ]:
#converting the near and far prices before 3:55 from 0 to Na
clean_full_df["near_price_clean"] = (
    clean_full_df["near_price"]
    .mask(clean_full_df["near_price"] == 0, np.nan)
    .astype(float)
)

clean_full_df["far_price_clean"] = (
    clean_full_df["far_price"]
    .mask(clean_full_df["far_price"] == 0, np.nan)
    .astype(float)
)

clean_full_df["near_move_bps"] = 10_000 * (
    clean_full_df["near_price_clean"]
    / clean_full_df["mid"]
    - 1
)

clean_full_df["far_move_bps"] = 10_000 * (
    clean_full_df["far_price_clean"]
    / clean_full_df["mid"]
    - 1
)

clean_full_df["near_far_spread_bps"] = 10_000 * (
    (
        clean_full_df["near_price_clean"]
        - clean_full_df["far_price_clean"]
    )
    / clean_full_df["mid"]
)


In [ ]:
print(
    clean_full_df[
        [
            "near_move_bps",
            "far_move_bps",
            "near_far_spread_bps",
            "cross_move_bps",
        ]
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)
print()

print(
    "Near available:",
    clean_full_df["near_price_clean"].notna().sum()
)

print(
    "Far available:",
    clean_full_df["far_price_clean"].notna().sum()
)

#rying to find out how the indicative prices provided by nasdaq line up with the actual cross
print(
    clean_full_df[
        [
            "near_move_bps",
            "far_move_bps",
            "near_far_spread_bps",
            "cross_move_bps",
        ]
    ].corr()
)

In [ ]:
#since there is a disparity between the number of near and far observations, this means that the previous assumption of far
#being 0 uptill 3:55 only was false. Far can be 0 after 3:55 as well but not for the same reasons as befroe 3:55pm

after_355 = clean_full_df[
    clean_full_df["local_time"].dt.time
    >= pd.Timestamp("15:55:00").time()
]

print(
    "Post-3:55 Near zeros:",
    (after_355["near_price"] == 0).sum()
)

print(
    "Post-3:55 Far zeros:",
    (after_355["far_price"] == 0).sum()
)
print()

print(
    after_355.loc[
        after_355["far_price"] == 0,
        [
            "date",
            "symbol",
            "local_time",
            "side",
            "shares",
            "paired_shares",
            "ref_price",
            "near_price",
            "far_price",
        ]
    ].head(20).to_string(index=False)
)

In [ ]:
#doing a seconds to close comparison of the MAE of near_move,cross_move,far_move and more
clean_full_df["seconds_to_close"] = (
    16 * 3600
    - (
        clean_full_df["local_time"].dt.hour * 3600
        + clean_full_df["local_time"].dt.minute * 60
        + clean_full_df["local_time"].dt.second
    )
)

horizons = [
    300,   # 5 mins
    240,
    180,
    120,
    60,
    30,
    10,
    5,
    1,
]

results = []

for h in horizons:
    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ]

    results.append({
        "seconds_to_close": h,
        "observations": len(g),

        "mid_mae_bps":
            g["cross_move_bps"].abs().mean(),

        "near_available":
            g["near_move_bps"].notna().sum(),

        "near_corr":
            g["near_move_bps"].corr(
                g["cross_move_bps"]
            ),

        "near_mae_bps":
            (
                g["near_move_bps"]
                - g["cross_move_bps"]
            ).abs().mean(),

        "far_available":
            g["far_move_bps"].notna().sum(),

        "far_corr":
            g["far_move_bps"].corr(
                g["cross_move_bps"]
            ),

        "far_mae_bps":
            (
                g["far_move_bps"]
                - g["cross_move_bps"]
            ).abs().mean(),
    })

horizon_summary = pd.DataFrame(results)

display(
    horizon_summary.round(3)
)

### 5.2 Signed auction imbalance: Construction and validation

In [ ]:
#checking to see if shares or adv is missing or 0 at points
print("Side values:")
print(clean_full_df["side"].value_counts(dropna=False))

print()

print("Missing shares:", clean_full_df["shares"].isna().sum())
print("Negative shares:", (clean_full_df["shares"] < 0).sum())

print("Missing ADV:", clean_full_df["adv"].isna().sum())
print("Non-positive ADV:", (clean_full_df["adv"] <= 0).sum())
print()

#adding the side for the share imbalance
side_sign = {
    "BUY": 1,
    "SELL": -1,
    "NONE": 0,
}

clean_full_df["side_sign"] = (
    clean_full_df["side"]
    .map(side_sign)
)

#construct the signed shares
clean_full_df["signed_imbalance_shares"] = (
    clean_full_df["side_sign"]
    * clean_full_df["shares"]
)

#normalize by adv
clean_full_df["signed_imbalance_adv"] = (
    clean_full_df["signed_imbalance_shares"]
    / (clean_full_df["adv"] * 1_000)
)

print(
    clean_full_df["signed_imbalance_adv"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)
print()

print(
    "Missing normalized imbalance:",
    clean_full_df["signed_imbalance_adv"].isna().sum()
)

In [ ]:
#checking insufficient rows
insufficient_rows = clean_full_df[
    clean_full_df["side"] == "INSUFFICIENT"
]

print("insuffiecient rows:", len(insufficient_rows))

print(
    insufficient_rows[
        [
            "date",
            "symbol",
            "local_time",
            "shares",
            "paired_shares",
            "ref_price",
            "near_price",
            "far_price",
            "seconds_to_close",
        ]
    ]
    .head(20)
    .to_string(index=False)
)
print()

print(
    insufficient_rows["seconds_to_close"]
    .describe()
)
print()

print(
    insufficient_rows["shares"]
    .describe()
)

In [ ]:
#checking the extremes of the normalized imbalances
extreme_imbalance = clean_full_df[
    clean_full_df["signed_imbalance_adv"].abs() > 0.50
]

print(
    "Rows with imbalance > 50% of ADV:",
    len(extreme_imbalance)
)
print()

print(
    extreme_imbalance[
        [
            "date",
            "symbol",
            "local_time",
            "side",
            "shares",
            "adv",
            "signed_imbalance_adv",
            "paired_shares",
            "cross_move_bps",
        ]
    ]
    .sort_values("signed_imbalance_adv")
    .head(20)
    .to_string(index=False)
)
print()

print(
    extreme_imbalance[
        [
            "date",
            "symbol",
            "local_time",
            "side",
            "shares",
            "adv",
            "signed_imbalance_adv",
            "paired_shares",
            "cross_move_bps",
        ]
    ]
    .sort_values(
        "signed_imbalance_adv",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

### 5.3 Signed imbalance: Predictive structure

In [ ]:
#checking signed imbalance predictive strength by time to close

imbalance_horizons = [
    600,   # 3:50
    300,
    240,
    180,
    120,
    60,
    30,
    10,
    5,
    1,
]

imbalance_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ]

    usable = g[
        [
            "signed_imbalance_adv",
            "cross_move_bps",
        ]
    ].dropna()

    imbalance_results.append({
        "seconds_to_close": h,

        "observations":
            len(g),

        "imbalance_available":
            len(usable),

        "imbalance_corr":
            usable["signed_imbalance_adv"].corr(
                usable["cross_move_bps"]
            ),

        "mean_abs_imbalance_pct_adv":
            100
            * usable["signed_imbalance_adv"]
            .abs()
            .mean(),
    })

imbalance_horizon_summary = pd.DataFrame(
    imbalance_results
)

display(
    imbalance_horizon_summary.round(4)
)

In [ ]:
#Due to PPC extreme values, we introduce Spearman correlation to check why there is a spike in correlation
#coefficient at 300s, in comparison to the Pearson correlation we have been using
imbalance_robustness_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ]

    usable = g[
        [
            "signed_imbalance_adv",
            "cross_move_bps",
        ]
    ].dropna()

    imbalance_robustness_results.append({
        "seconds_to_close": h,

        "observations":
            len(usable),

        "pearson_corr":
            usable["signed_imbalance_adv"].corr(
                usable["cross_move_bps"],
                method="pearson",
            ),

      # spearman helps us check whether relationship is broadly monotonic or
      # just being dependent on a few huge numerical observations.
        "spearman_corr":
            usable["signed_imbalance_adv"].corr(
                usable["cross_move_bps"],
                method="spearman",
            ),
    })

imbalance_robustness_summary = pd.DataFrame(
    imbalance_robustness_results
)

display(
    imbalance_robustness_summary.round(4)
)

In [ ]:
# creating imbalance deciles one second before the close to check if the monotonic structure
# shown by the spearman correlation is real

h = 1

g = clean_full_df[
    clean_full_df["seconds_to_close"] == h
][
    [
        "signed_imbalance_adv",
        "cross_move_bps",
    ]
].dropna().copy()

zero_imbalance = g[
    g["signed_imbalance_adv"] == 0
]

nonzero_imbalance = g[
    g["signed_imbalance_adv"] != 0
].copy()

nonzero_imbalance["imbalance_decile"] = (
    pd.qcut(
        nonzero_imbalance["signed_imbalance_adv"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1
)

imbalance_decile_summary = (
    nonzero_imbalance
    .groupby("imbalance_decile")
    .agg(
        observations=(
            "cross_move_bps",
            "size",
        ),

        min_imbalance_pct_adv=(
            "signed_imbalance_adv",
            lambda s: 100 * s.min(),
        ),

        max_imbalance_pct_adv=(
            "signed_imbalance_adv",
            lambda s: 100 * s.max(),
        ),

        mean_imbalance_pct_adv=(
            "signed_imbalance_adv",
            lambda s: 100 * s.mean(),
        ),

        mean_cross_move_bps=(
            "cross_move_bps",
            "mean",
        ),

        median_cross_move_bps=(
            "cross_move_bps",
            "median",
        ),
    )
    .reset_index()
)

print(
    "Zero imbalance observations:",
    len(zero_imbalance),
)

print(
    "Mean cross move for zero imbalance:",
    zero_imbalance["cross_move_bps"].mean(),
)

print(
    "Median cross move for zero imbalance:",
    zero_imbalance["cross_move_bps"].median(),
)

display(
    imbalance_decile_summary.round(4)
)

In [ ]:
# checking direction versus magnitude at one second to close

h = 1

g = clean_full_df[
    clean_full_df["seconds_to_close"] == h
][
    [
        "side",
        "signed_imbalance_adv",
        "cross_move_bps",
    ]
].dropna().copy()


# checking the target behavior by auction imbalance direction

side_summary = (
    g
    .groupby("side")
    .agg(
        observations=(
            "cross_move_bps",
            "size",
        ),

        mean_imbalance_pct_adv=(
            "signed_imbalance_adv",
            lambda s: 100 * s.mean(),
        ),

        mean_cross_move_bps=(
            "cross_move_bps",
            "mean",
        ),

        median_cross_move_bps=(
            "cross_move_bps",
            "median",
        ),
    )
    .reset_index()
)

display(
    side_summary.round(4)
)
print()

# does magnitude still matter within buy and sell

within_side_results = []

for side_value in ["SELL", "BUY"]:

    s = g[
        g["side"] == side_value
    ]

    within_side_results.append({
        "side":
            side_value,

        "observations":
            len(s),

        "pearson_corr":
            s["signed_imbalance_adv"].corr(
                s["cross_move_bps"],
                method="pearson",
            ),

        "spearman_corr":
            s["signed_imbalance_adv"].corr(
                s["cross_move_bps"],
                method="spearman",
            ),
    })

within_side_summary = pd.DataFrame(
    within_side_results
)

display(
    within_side_summary.round(4)
)

In [ ]:
#checking the direction versus magnitude across all horizons

direction_horizon_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "cross_move_bps",
        ]
    ].dropna().copy()

    sell = g[
        g["side"] == "SELL"
    ]

    none = g[
        g["side"] == "NONE"
    ]

    buy = g[
        g["side"] == "BUY"
    ]

    direction_horizon_results.append({

        "seconds_to_close":
            h,

        "sell_observations":
            len(sell),

        "sell_mean_cross_bps":
            sell["cross_move_bps"].mean(),

        "sell_spearman":
            sell["signed_imbalance_adv"].corr(
                sell["cross_move_bps"],
                method="spearman",
            ),

        "none_observations":
            len(none),

        "none_mean_cross_bps":
            none["cross_move_bps"].mean(),

        "buy_observations":
            len(buy),

        "buy_mean_cross_bps":
            buy["cross_move_bps"].mean(),

        "buy_spearman":
            buy["signed_imbalance_adv"].corr(
                buy["cross_move_bps"],
                method="spearman",
            ),
    })

direction_horizon_summary = pd.DataFrame(
    direction_horizon_results
)

display(
    direction_horizon_summary.round(4)
)

In [ ]:
# checking directional effect relative to the none state by horizon

direction_effect_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "cross_move_bps",
        ]
    ].dropna().copy()

    sell = g[
        g["side"] == "SELL"
    ]

    none = g[
        g["side"] == "NONE"
    ]

    buy = g[
        g["side"] == "BUY"
    ]

    total = len(g)

    sell_mean = sell["cross_move_bps"].mean()
    none_mean = none["cross_move_bps"].mean()
    buy_mean = buy["cross_move_bps"].mean()

    sell_median = sell["cross_move_bps"].median()
    none_median = none["cross_move_bps"].median()
    buy_median = buy["cross_move_bps"].median()

    direction_effect_results.append({

        "seconds_to_close":
            h,

        "sell_share_pct":
            100 * len(sell) / total,

        "none_share_pct":
            100 * len(none) / total,

        "buy_share_pct":
            100 * len(buy) / total,

        "sell_minus_none_mean_bps":
            sell_mean - none_mean,

        "buy_minus_none_mean_bps":
            buy_mean - none_mean,

        "buy_minus_sell_mean_bps":
            buy_mean - sell_mean,

        "sell_minus_none_median_bps":
            sell_median - none_median,

        "buy_minus_none_median_bps":
            buy_median - none_median,
    })

direction_effect_summary = pd.DataFrame(
    direction_effect_results
)

display(
    direction_effect_summary.round(4)
)

### 5.4 Incremental information beyond the Near price

In [ ]:
# checking if imbalance direction explain Near residual error?

late_horizons = [
    120,
    60,
    30,
    10,
    5,
    1,
]

near_residual_results = []

for h in late_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "near_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["near_residual_bps"] = (
        g["cross_move_bps"]
        - g["near_move_bps"]
    )

    sell = g[
        g["side"] == "SELL"
    ]

    none = g[
        g["side"] == "NONE"
    ]

    buy = g[
        g["side"] == "BUY"
    ]

    sell_mean = sell[
        "near_residual_bps"
    ].mean()

    none_mean = none[
        "near_residual_bps"
    ].mean()

    buy_mean = buy[
        "near_residual_bps"
    ].mean()

    sell_median = sell[
        "near_residual_bps"
    ].median()

    none_median = none[
        "near_residual_bps"
    ].median()

    buy_median = buy[
        "near_residual_bps"
    ].median()

    near_residual_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "sell_mean_residual_bps":
            sell_mean,

        "none_mean_residual_bps":
            none_mean,

        "buy_mean_residual_bps":
            buy_mean,

        "sell_minus_none_mean_bps":
            sell_mean - none_mean,

        "buy_minus_none_mean_bps":
            buy_mean - none_mean,

        "sell_median_residual_bps":
            sell_median,

        "none_median_residual_bps":
            none_median,

        "buy_median_residual_bps":
            buy_median,
    })

near_residual_summary = pd.DataFrame(
    near_residual_results
)

display(
    near_residual_summary.round(4)
)

In [ ]:
# checking if the magnitude of imbalance explains Near residuals within side

near_residual_magnitude_results = []

for h in late_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "near_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["near_residual_bps"] = (
        g["cross_move_bps"]
        - g["near_move_bps"]
    )

    g["abs_imbalance_adv"] = (
        g["signed_imbalance_adv"].abs()
    )

    for side_value in [
        "SELL",
        "BUY",
    ]:

        s = g[
            g["side"] == side_value
        ]

        near_residual_magnitude_results.append({

            "seconds_to_close":
                h,

            "side":
                side_value,

            "observations":
                len(s),

            "pearson_corr":
                s["abs_imbalance_adv"].corr(
                    s["near_residual_bps"],
                    method="pearson",
                ),

            "spearman_corr":
                s["abs_imbalance_adv"].corr(
                    s["near_residual_bps"],
                    method="spearman",
                ),
        })

near_residual_magnitude_summary = pd.DataFrame(
    near_residual_magnitude_results
)

display(
    near_residual_magnitude_summary.round(4)
)

In [ ]:
#checking the shape of imbalance magnitude versus Near residual to find more about their relationship
shape_horizons = [
    120,
    30,
    1,
]

near_residual_shape_results = []

for h in shape_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "near_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["near_residual_bps"] = (
        g["cross_move_bps"]
        - g["near_move_bps"]
    )

    g["abs_imbalance_adv"] = (
        g["signed_imbalance_adv"].abs()
    )

    for side_value in [
        "SELL",
        "BUY",
    ]:

        s = g[
            g["side"] == side_value
        ].copy()

        s["magnitude_quintile"] = pd.qcut(
            s["abs_imbalance_adv"],
            q=5,
            labels=[
                "Q1",
                "Q2",
                "Q3",
                "Q4",
                "Q5",
            ],
            duplicates="drop",
        )

        summary = (
            s
            .groupby(
                "magnitude_quintile",
                observed=True,
            )
            .agg(
                observations=(
                    "near_residual_bps",
                    "size",
                ),

                mean_abs_imbalance_pct_adv=(
                    "abs_imbalance_adv",
                    lambda x: 100 * x.mean(),
                ),

                mean_near_residual_bps=(
                    "near_residual_bps",
                    "mean",
                ),

                median_near_residual_bps=(
                    "near_residual_bps",
                    "median",
                ),
            )
            .reset_index()
        )

        summary[
            "seconds_to_close"
        ] = h

        summary[
            "side"
        ] = side_value

        near_residual_shape_results.append(
            summary
        )

near_residual_shape_summary = pd.concat(
    near_residual_shape_results,
    ignore_index=True,
)

near_residual_shape_summary = (
    near_residual_shape_summary[
        [
            "seconds_to_close",
            "side",
            "magnitude_quintile",
            "observations",
            "mean_abs_imbalance_pct_adv",
            "mean_near_residual_bps",
            "median_near_residual_bps",
        ]
    ]
)

display(
    near_residual_shape_summary.round(4)
)

### 5.5 Paired auction volume

In [ ]:
#checking paired auction shares

paired_audit = pd.DataFrame({

    "observations": [
        len(clean_full_df)
    ],

    "missing_paired_shares": [
        clean_full_df[
            "paired_shares"
        ].isna().sum()
    ],

    "negative_paired_shares": [
        (
            clean_full_df[
                "paired_shares"
            ] < 0
        ).sum()
    ],

    "zero_paired_shares": [
        (
            clean_full_df[
                "paired_shares"
            ] == 0
        ).sum()
    ],
})

display(
    paired_audit
)


display(
    clean_full_df[
        "paired_shares"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)


paired_horizon_summary = (
    clean_full_df[
        clean_full_df[
            "seconds_to_close"
        ].isin(
            imbalance_horizons
        )
    ]
    .groupby(
        "seconds_to_close"
    )
    .agg(
        observations=(
            "paired_shares",
            "size",
        ),

        mean_paired_shares=(
            "paired_shares",
            "mean",
        ),

        median_paired_shares=(
            "paired_shares",
            "median",
        ),

        zero_share_pct=(
            "paired_shares",
            lambda x: 100 * (x == 0).mean(),
        ),
    )
    .reset_index()
    .sort_values(
        "seconds_to_close",
        ascending=False,
    )
)

display(
    paired_horizon_summary.round(4)
)

In [ ]:
#normalizing paired auction shares by ADV

clean_full_df["paired_adv"] = (
    clean_full_df["paired_shares"]
    / (
        clean_full_df["adv"]
        * 1000
    )
)


display(
    clean_full_df[
        "paired_adv"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)


paired_adv_horizon_summary = (
    clean_full_df[
        clean_full_df[
            "seconds_to_close"
        ].isin(
            imbalance_horizons
        )
    ]
    .groupby(
        "seconds_to_close"
    )
    .agg(
        observations=(
            "paired_adv",
            "size",
        ),

        mean_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.mean(),
        ),

        median_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.median(),
        ),

        p95_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.quantile(0.95),
        ),
    )
    .reset_index()
    .sort_values(
        "seconds_to_close",
        ascending=False,
    )
)

print()
display(
    paired_adv_horizon_summary.round(4)
)

In [ ]:
#checking paired auction volume above 100% of ADV

paired_extreme = clean_full_df[
    clean_full_df["paired_adv"] > 1
][
    [
        "date",
        "symbol",
        "local_time",
        "seconds_to_close",
        "paired_shares",
        "adv",
        "paired_adv",
    ]
].copy()


paired_extreme_audit = pd.DataFrame({

    "rows_over_100pct_adv": [
        len(paired_extreme)
    ],

    "pct_of_all_rows": [
        100
        * len(paired_extreme)
        / len(clean_full_df)
    ],

    "stock_days_over_100pct_adv": [
        paired_extreme[
            [
                "date",
                "symbol",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    ],
})

display(
    paired_extreme_audit.round(4)
)


extreme_stock_days = (
    paired_extreme
    .groupby(
        [
            "date",
            "symbol",
        ]
    )
    .agg(
        rows_over_100pct_adv=(
            "paired_adv",
            "size",
        ),

        max_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.max(),
        ),

        median_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.median(),
        ),

        earliest_seconds_to_close=(
            "seconds_to_close",
            "max",
        ),

        latest_seconds_to_close=(
            "seconds_to_close",
            "min",
        ),
    )
    .reset_index()
    .sort_values(
        "max_paired_pct_adv",
        ascending=False,
    )
    .reset_index(
      drop=True
    )
)

print()
display(
    extreme_stock_days
    .head(20)
    .round(4)
)

In [ ]:
# checking date concentration of extreme paired auction volume

extreme_by_date = (
    paired_extreme
    .groupby(
        "date"
    )
    .agg(
        rows_over_100pct_adv=(
            "paired_adv",
            "size",
        ),

        stock_days_over_100pct_adv=(
            "symbol",
            "nunique",
        ),

        max_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.max(),
        ),

        median_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.median(),
        ),
    )
    .reset_index()
    .sort_values(
        "stock_days_over_100pct_adv",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

display(
    extreme_by_date.round(4)
)

In [ ]:
#comparing paired auction volume across dates
paired_date_horizons = [
    120,
    30,
    1,
]

paired_by_date = (
    clean_full_df[
        clean_full_df[
            "seconds_to_close"
        ].isin(
            paired_date_horizons
        )
    ]
    .groupby(
        [
            "date",
            "seconds_to_close",
        ]
    )
    .agg(
        observations=(
            "paired_shares",
            "size",
        ),

        median_paired_shares=(
            "paired_shares",
            "median",
        ),

        mean_paired_shares=(
            "paired_shares",
            "mean",
        ),

        median_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.median(),
        ),

        mean_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.mean(),
        ),

        p95_paired_pct_adv=(
            "paired_adv",
            lambda x: 100 * x.quantile(0.95),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "seconds_to_close",
            "median_paired_pct_adv",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

display(
    paired_by_date.round(4))

In [ ]:
#checking the predictive structure of paired auction volume
paired_predictive_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "paired_adv",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["abs_cross_move_bps"] = (
        g["cross_move_bps"].abs()
    )

    paired_predictive_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "signed_pearson":
            g["paired_adv"].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "signed_spearman":
            g["paired_adv"].corr(
                g["cross_move_bps"],
                method="spearman",
            ),

        "absolute_pearson":
            g["paired_adv"].corr(
                g["abs_cross_move_bps"],
                method="pearson",
            ),

        "absolute_spearman":
            g["paired_adv"].corr(
                g["abs_cross_move_bps"],
                method="spearman",
            ),
    })

paired_predictive_summary = pd.DataFrame(
    paired_predictive_results
)

display(
    paired_predictive_summary.round(4)
)

In [ ]:
# Check robustness after excluding the date with the highest median paired volume.
paired_stress_date = (
    clean_full_df
    .groupby("date")["paired_adv"]
    .median()
    .idxmax()
)

paired_robustness_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "date",
            "paired_adv",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["abs_cross_move_bps"] = (
        g["cross_move_bps"].abs()
    )

    g_ex_stress_date = g[
        g["date"] != paired_stress_date
    ].copy()

    paired_robustness_results.append({

        "seconds_to_close":
            h,

        "all_observations":
            len(g),

        "ex_stress_date_observations":
            len(g_ex_stress_date),

        "all_absolute_pearson":
            g["paired_adv"].corr(
                g["abs_cross_move_bps"],
                method="pearson",
            ),

        "ex_stress_date_absolute_pearson":
            g_ex_stress_date["paired_adv"].corr(
                g_ex_stress_date["abs_cross_move_bps"],
                method="pearson",
            ),

        "all_absolute_spearman":
            g["paired_adv"].corr(
                g["abs_cross_move_bps"],
                method="spearman",
            ),

        "ex_stress_date_absolute_spearman":
            g_ex_stress_date["paired_adv"].corr(
                g_ex_stress_date["abs_cross_move_bps"],
                method="spearman",
            ),
    })

paired_robustness_summary = pd.DataFrame(
    paired_robustness_results
)

display(
    paired_robustness_summary.round(4)
)

### 5.6 Auction relative imbalance

In [ ]:
#constructing and checking the auction relative imbalance

clean_full_df["auction_total_shares"] = (
    clean_full_df["paired_shares"]
    + clean_full_df["shares"]
)

clean_full_df["signed_auction_imbalance_ratio"] = np.where(
    clean_full_df["auction_total_shares"] > 0,
    clean_full_df["signed_imbalance_shares"]
    / clean_full_df["auction_total_shares"],
    np.nan,
)


auction_ratio_audit = pd.DataFrame({

    "observations": [
        len(clean_full_df)
    ],

    "zero_total_auction_shares": [
        (
            clean_full_df[
                "auction_total_shares"
            ] == 0
        ).sum()
    ],

    "missing_ratio": [
        clean_full_df[
            "signed_auction_imbalance_ratio"
        ].isna().sum()
    ],

    "below_minus_one": [
        (
            clean_full_df[
                "signed_auction_imbalance_ratio"
            ] < -1
        ).sum()
    ],

    "above_plus_one": [
        (
            clean_full_df[
                "signed_auction_imbalance_ratio"
            ] > 1
        ).sum()
    ],
})

display(
    auction_ratio_audit
)


display(
    clean_full_df[
        "signed_auction_imbalance_ratio"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
#comparing ADV and auction relative imbalance normalization to see which better captures
#relation between imbalance and the closing cross

imbalance_normalization_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "signed_imbalance_adv",
            "signed_auction_imbalance_ratio",
            "cross_move_bps",
        ]
    ].dropna().copy()

    imbalance_normalization_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "adv_pearson":
            g["signed_imbalance_adv"].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "adv_spearman":
            g["signed_imbalance_adv"].corr(
                g["cross_move_bps"],
                method="spearman",
            ),

        "auction_ratio_pearson":
            g[
                "signed_auction_imbalance_ratio"
            ].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "auction_ratio_spearman":
            g[
                "signed_auction_imbalance_ratio"
            ].corr(
                g["cross_move_bps"],
                method="spearman",
            ),
    })

imbalance_normalization_summary = pd.DataFrame(
    imbalance_normalization_results
)

display(
    imbalance_normalization_summary.round(4)
)

In [ ]:
#comparing within side magnitude for ADV and auction relative normalization
within_side_normalization_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "signed_auction_imbalance_ratio",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["abs_imbalance_adv"] = (
        g["signed_imbalance_adv"].abs()
    )

    g["abs_auction_imbalance_ratio"] = (
        g["signed_auction_imbalance_ratio"].abs()
    )

    for side_value in [
        "SELL",
        "BUY",
    ]:

        s = g[
            g["side"] == side_value
        ]

        within_side_normalization_results.append({

            "seconds_to_close":
                h,

            "side":
                side_value,

            "observations":
                len(s),

            "adv_pearson":
                s["abs_imbalance_adv"].corr(
                    s["cross_move_bps"],
                    method="pearson",
                ),

            "adv_spearman":
                s["abs_imbalance_adv"].corr(
                    s["cross_move_bps"],
                    method="spearman",
                ),

            "auction_ratio_pearson":
                s[
                    "abs_auction_imbalance_ratio"
                ].corr(
                    s["cross_move_bps"],
                    method="pearson",
                ),

            "auction_ratio_spearman":
                s[
                    "abs_auction_imbalance_ratio"
                ].corr(
                    s["cross_move_bps"],
                    method="spearman",
                ),
        })

within_side_normalization_summary = pd.DataFrame(
    within_side_normalization_results
)

display(
    within_side_normalization_summary.round(4)
)

In [ ]:
#comparing ADV and auction relative magnitude for explaining Near residuals within side
near_residual_normalization_results = []

for h in late_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "side",
            "signed_imbalance_adv",
            "signed_auction_imbalance_ratio",
            "near_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["near_residual_bps"] = (
        g["cross_move_bps"]
        - g["near_move_bps"]
    )

    g["abs_imbalance_adv"] = (
        g["signed_imbalance_adv"].abs()
    )

    g["abs_auction_imbalance_ratio"] = (
        g["signed_auction_imbalance_ratio"].abs()
    )

    for side_value in [
        "SELL",
        "BUY",
    ]:

        s = g[
            g["side"] == side_value
        ]

        near_residual_normalization_results.append({

            "seconds_to_close":
                h,

            "side":
                side_value,

            "observations":
                len(s),

            "adv_pearson":
                s["abs_imbalance_adv"].corr(
                    s["near_residual_bps"],
                    method="pearson",
                ),

            "adv_spearman":
                s["abs_imbalance_adv"].corr(
                    s["near_residual_bps"],
                    method="spearman",
                ),

            "auction_ratio_pearson":
                s[
                    "abs_auction_imbalance_ratio"
                ].corr(
                    s["near_residual_bps"],
                    method="pearson",
                ),

            "auction_ratio_spearman":
                s[
                    "abs_auction_imbalance_ratio"
                ].corr(
                    s["near_residual_bps"],
                    method="spearman",
                ),
        })

near_residual_normalization_summary = pd.DataFrame(
    near_residual_normalization_results
)

display(
    near_residual_normalization_summary.round(4)
)

### 5.7 Reference price displacement

In [ ]:
#checking the auction reference price

ref_price_audit = pd.DataFrame({

    "observations": [
        len(clean_full_df)
    ],

    "missing_ref_price": [
        clean_full_df[
            "ref_price"
        ].isna().sum()
    ],

    "zero_ref_price": [
        (
            clean_full_df[
                "ref_price"
            ] == 0
        ).sum()
    ],

    "nonpositive_ref_price": [
        (
            clean_full_df[
                "ref_price"
            ] <= 0
        ).sum()
    ],
})

display(
    ref_price_audit
)


ref_horizon_summary = (
    clean_full_df[
        clean_full_df[
            "seconds_to_close"
        ].isin(
            imbalance_horizons
        )
    ]
    .groupby(
        "seconds_to_close"
    )
    .agg(
        observations=(
            "ref_price",
            "size",
        ),

        available_ref_price=(
            "ref_price",
            lambda x: x.notna().sum(),
        ),

        zero_ref_price=(
            "ref_price",
            lambda x: (x == 0).sum(),
        ),

        median_ref_price=(
            "ref_price",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        "seconds_to_close",
        ascending=False,
    )
)

print()
display(
    ref_horizon_summary
)

In [ ]:
#checking zero reference price states and constructing reference price displacement
ref_zero = (
    clean_full_df["ref_price"] <= 0
)

insufficient_state = (
    clean_full_df["side"] == "INSUFFICIENT"
)

zero_auction_total = (
    clean_full_df["auction_total_shares"] == 0
)


ref_zero_overlap_audit = pd.DataFrame({

    "zero_ref_price_rows": [
        ref_zero.sum()
    ],

    "insufficient_rows": [
        insufficient_state.sum()
    ],

    "zero_auction_total_rows": [
        zero_auction_total.sum()
    ],

    "zero_ref_and_insufficient": [
        (
            ref_zero
            & insufficient_state
        ).sum()
    ],

    "zero_ref_not_insufficient": [
        (
            ref_zero
            & ~insufficient_state
        ).sum()
    ],

    "insufficient_not_zero_ref": [
        (
            insufficient_state
            & ~ref_zero
        ).sum()
    ],

    "zero_ref_and_zero_auction_total": [
        (
            ref_zero
            & zero_auction_total
        ).sum()
    ],
})

display(
    ref_zero_overlap_audit
)


clean_full_df["ref_price_clean"] = (
    clean_full_df["ref_price"]
    .mask(
        ref_zero,
        np.nan,
    )
)


clean_full_df["ref_move_bps"] = (
    10000
    * (
        clean_full_df["ref_price_clean"]
        / clean_full_df["mid"]
        - 1
    )
)

print()
display(
    clean_full_df[
        "ref_move_bps"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
#checking the fixed horizon predictive value of ref price

ref_predictive_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "ref_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    midpoint_mae = (
        g["cross_move_bps"]
        .abs()
        .mean()
    )

    ref_mae = (
        (
            g["cross_move_bps"]
            - g["ref_move_bps"]
        )
        .abs()
        .mean()
    )

    ref_predictive_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "ref_pearson":
            g["ref_move_bps"].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "ref_spearman":
            g["ref_move_bps"].corr(
                g["cross_move_bps"],
                method="spearman",
            ),

        "midpoint_mae_bps":
            midpoint_mae,

        "ref_mae_bps":
            ref_mae,

        "ref_mae_improvement_bps":
            midpoint_mae - ref_mae,
    })

ref_predictive_summary = pd.DataFrame(
    ref_predictive_results
)

display(
    ref_predictive_summary.round(4)
)

In [ ]:
# checking if reference price explain Near's residual error
ref_near_results = []

for h in late_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "ref_move_bps",
            "near_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    g["near_residual_bps"] = (
        g["cross_move_bps"]
        - g["near_move_bps"]
    )

    g["ref_minus_near_bps"] = (
        g["ref_move_bps"]
        - g["near_move_bps"]
    )

    near_mae = (
        g["near_residual_bps"]
        .abs()
        .mean()
    )

    ref_mae = (
        (
            g["cross_move_bps"]
            - g["ref_move_bps"]
        )
        .abs()
        .mean()
    )

    ref_near_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "ref_minus_near_pearson":
            g["ref_minus_near_bps"].corr(
                g["near_residual_bps"],
                method="pearson",
            ),

        "ref_minus_near_spearman":
            g["ref_minus_near_bps"].corr(
                g["near_residual_bps"],
                method="spearman",
            ),

        "near_mae_bps":
            near_mae,

        "ref_mae_bps":
            ref_mae,
    })

ref_near_summary = pd.DataFrame(
    ref_near_results
)

display(
    ref_near_summary.round(4)
)

In [ ]:
#due to mathematical coupling of having the -near component in both residuals,
#doing a partial correlation of reference and target after controlling for Near

ref_partial_results = []

for h in late_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "near_move_bps",
            "ref_move_bps",
            "cross_move_bps",
        ]
    ].dropna().copy()

    # Fit target as a linear function of Near
    near_target_coef = np.polyfit(
        g["near_move_bps"],
        g["cross_move_bps"],
        deg=1,
    )

    target_fitted_from_near = np.polyval(
        near_target_coef,
        g["near_move_bps"],
    )

    g["target_residual_after_near"] = (
        g["cross_move_bps"]
        - target_fitted_from_near
    )

    # Fit Reference as a linear function of Near
    near_ref_coef = np.polyfit(
        g["near_move_bps"],
        g["ref_move_bps"],
        deg=1,
    )

    ref_fitted_from_near = np.polyval(
        near_ref_coef,
        g["near_move_bps"],
    )

    g["ref_residual_after_near"] = (
        g["ref_move_bps"]
        - ref_fitted_from_near
    )

    ref_partial_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "partial_pearson":
            g[
                "ref_residual_after_near"
            ].corr(
                g["target_residual_after_near"],
                method="pearson",
            ),

        "partial_spearman":
            g[
                "ref_residual_after_near"
            ].corr(
                g["target_residual_after_near"],
                method="spearman",
            ),
    })

ref_partial_summary = pd.DataFrame(
    ref_partial_results
)

display(
    ref_partial_summary.round(4)
)

### 5.8 Regular market context

In [ ]:
#constructing and inspecting regular market context features

clean_full_df["spread_bps"] = (
    10000
    * (
        clean_full_df["ask"]
        - clean_full_df["bid"]
    )
    / clean_full_df["mid"]
)


book_depth_total = (
    clean_full_df["bid_qty"]
    + clean_full_df["ask_qty"]
)

clean_full_df["book_imbalance"] = np.where(
    book_depth_total > 0,
    (
        clean_full_df["bid_qty"]
        - clean_full_df["ask_qty"]
    )
    / book_depth_total,
    np.nan,
)


clean_full_df["open_to_mid_bps"] = (
    10000
    * (
        clean_full_df["mid"]
        / clean_full_df["open"]
        - 1
    )
)


market_context_audit = pd.DataFrame({

    "observations": [
        len(clean_full_df)
    ],

    "missing_spread": [
        clean_full_df[
            "spread_bps"
        ].isna().sum()
    ],

    "negative_spread": [
        (
            clean_full_df[
                "spread_bps"
            ] < 0
        ).sum()
    ],

    "zero_book_depth": [
        (
            book_depth_total
            <= 0
        ).sum()
    ],

    "missing_book_imbalance": [
        clean_full_df[
            "book_imbalance"
        ].isna().sum()
    ],

    "book_below_minus_one": [
        (
            clean_full_df[
                "book_imbalance"
            ] < -1
        ).sum()
    ],

    "book_above_plus_one": [
        (
            clean_full_df[
                "book_imbalance"
            ] > 1
        ).sum()
    ],

    "missing_open_to_mid": [
        clean_full_df[
            "open_to_mid_bps"
        ].isna().sum()
    ],
})

display(
    market_context_audit
)

print()
display(
    clean_full_df[
        [
            "spread_bps",
            "book_imbalance",
            "open_to_mid_bps",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
#checking the non positive opening prices we foud out from previous result
open_problem = (
    clean_full_df["open"] <= 0
)

nonfinite_open_move = (
    ~np.isfinite(
        clean_full_df["open_to_mid_bps"]
    )
)


open_problem_audit = pd.DataFrame({

    "observations": [
        len(clean_full_df)
    ],

    "missing_open_price": [
        clean_full_df[
            "open"
        ].isna().sum()
    ],

    "zero_open_price": [
        (
            clean_full_df[
                "open"
            ] == 0
        ).sum()
    ],

    "negative_open_price": [
        (
            clean_full_df[
                "open"
            ] < 0
        ).sum()
    ],

    "nonfinite_open_to_mid": [
        nonfinite_open_move.sum()
    ],

    "stock_days_with_nonpositive_open": [
        clean_full_df.loc[
            open_problem,
            [
                "date",
                "symbol",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    ],
})

display(
    open_problem_audit
)
print()

open_problem_stock_days = (
    clean_full_df.loc[
        open_problem,
        [
            "date",
            "symbol",
            "open",
            "mid",
            "seconds_to_close",
        ]
    ]
    .groupby(
        [
            "date",
            "symbol",
        ]
    )
    .agg(
        observations=(
            "open",
            "size",
        ),

        min_open=(
            "open",
            "min",
        ),

        max_open=(
            "open",
            "max",
        ),

        median_mid=(
            "mid",
            "median",
        ),

        earliest_seconds_to_close=(
            "seconds_to_close",
            "max",
        ),

        latest_seconds_to_close=(
            "seconds_to_close",
            "min",
        ),
    )
    .reset_index()
    .sort_values(
        "observations",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

display(
    open_problem_stock_days
    .head(20)
    .round(4)
)

In [ ]:
#cleaning the opening price for the 2 IWM stock day zero observations and
#testing the regular market features at fixed horizons

clean_full_df["open_price_clean"] = (
    clean_full_df["open"]
    .mask(
        clean_full_df["open"] <= 0,
        np.nan,
    )
)

clean_full_df["open_to_mid_bps"] = (
    10000
    * (
        clean_full_df["mid"]
        / clean_full_df["open_price_clean"]
        - 1
    )
)


market_context_results = []

for h in imbalance_horizons:

    g = clean_full_df[
        clean_full_df["seconds_to_close"] == h
    ][
        [
            "spread_bps",
            "book_imbalance",
            "open_to_mid_bps",
            "cross_move_bps",
        ]
    ].copy()

    g["abs_cross_move_bps"] = (
        g["cross_move_bps"].abs()
    )

    market_context_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "open_available":
            g["open_to_mid_bps"]
            .notna()
            .sum(),

        "spread_abs_pearson":
            g["spread_bps"].corr(
                g["abs_cross_move_bps"],
                method="pearson",
            ),

        "spread_abs_spearman":
            g["spread_bps"].corr(
                g["abs_cross_move_bps"],
                method="spearman",
            ),

        "book_signed_pearson":
            g["book_imbalance"].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "book_signed_spearman":
            g["book_imbalance"].corr(
                g["cross_move_bps"],
                method="spearman",
            ),

        "open_signed_pearson":
            g["open_to_mid_bps"].corr(
                g["cross_move_bps"],
                method="pearson",
            ),

        "open_signed_spearman":
            g["open_to_mid_bps"].corr(
                g["cross_move_bps"],
                method="spearman",
            ),
    })

market_context_summary = pd.DataFrame(
    market_context_results
)

print()
display(
    market_context_summary.round(4)
)

## 6. Modeling and chronological evaluation

After exploring the individual auction signals, I moved to the prediction problem itself. I used fixed horizons of 120, 60, 30, 10, 5, and 1 seconds before the close, with one observation per stock-day at each horizon.

The data is split chronologically rather than randomly. The first 13 trading days are used for training, the next 4 for validation, and the final 4 days are kept completely separate as the test set. The test period is not used while choosing features, models, or hyperparameters.

I started with simple market-price baselines and linear models before moving to Ridge and XGBoost. This makes it possible to see whether the extra complexity is actually adding anything. The validation results are then used to freeze the final model specification before it is evaluated once on the held-out test period.

### 6.1 Modeling sample and chronological split

In [ ]:
#constructing fixed horizon modeling sample and assigning chronological train,validation,test splits

model_horizons = [
    120,
    60,
    30,
    10,
    5,
    1,
]

model_features = [
    "near_move_bps",
    "ref_move_bps",
    "signed_auction_imbalance_ratio",
    "signed_imbalance_adv",
    "side_sign",
    "paired_adv",
    "spread_bps",
]

model_columns = (
    [
        "date",
        "symbol",
        "seconds_to_close",
        "cross_move_bps",
    ]
    + model_features
)


modeling_df = (
    clean_full_df[
        clean_full_df["seconds_to_close"].isin(
            model_horizons
        )
    ][
        model_columns
    ]
    .copy()
)


all_model_dates = sorted(
    modeling_df["date"].unique()
)

train_dates = all_model_dates[:13]
validation_dates = all_model_dates[13:17]
test_dates = all_model_dates[17:]


modeling_df["split"] = np.select(
    [
        modeling_df["date"].isin(
            train_dates
        ),

        modeling_df["date"].isin(
            validation_dates
        ),

        modeling_df["date"].isin(
            test_dates
        ),
    ],
    [
        "train",
        "validation",
        "test",
    ],
    default="unassigned",
)


print("Training dates:")
print(train_dates)

print()

print("Validation dates:")
print(validation_dates)

print()

print("Test dates:")
print(test_dates)

print()


print(
    "Duplicate date-symbol-horizon rows:",
    modeling_df.duplicated(
        subset=[
            "date",
            "symbol",
            "seconds_to_close",
        ]
    ).sum()
)

print(
    "Unassigned rows:",
    (
        modeling_df["split"]
        == "unassigned"
    ).sum()
)

print()


split_summary = (
    modeling_df
    .groupby(
        [
            "split",
            "seconds_to_close",
        ]
    )
    .agg(
        observations=(
            "cross_move_bps",
            "size",
        ),

        stock_days=(
            "symbol",
            "size",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "seconds_to_close",
            "split",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

print()
display(
    split_summary
)


missing_by_feature = (
    modeling_df[
        model_features
    ]
    .isna()
    .sum()
    .rename(
        "missing_values"
    )
    .to_frame()
)

print()
display(
    missing_by_feature
)

### 6.2 Baselines and evaluation framework

In [ ]:
#evaluating simple market baselines on training and validation data
baseline_results = []

for split_value in [
    "train",
    "validation",
]:

    for h in model_horizons:

        g = modeling_df[
            (
                modeling_df["split"]
                == split_value
            )
            & (
                modeling_df["seconds_to_close"]
                == h
            )
        ].copy()

        y = g[
            "cross_move_bps"
        ]

        baseline_predictions = {

            "Midpoint":
                np.zeros(
                    len(g)
                ),

            "Reference":
                g["ref_move_bps"],

            "Near":
                g["near_move_bps"],
        }

        for model_name, y_pred in (
            baseline_predictions.items()
        ):

            baseline_results.append({

                "split":
                    split_value,

                "seconds_to_close":
                    h,

                "model":
                    model_name,

                "observations":
                    len(g),

                "mae_bps":
                    mean_absolute_error(
                        y,
                        y_pred,
                    ),

                "rmse_bps":
                    np.sqrt(
                        mean_squared_error(
                            y,
                            y_pred,
                        )
                    ),

                "pearson":
                    pd.Series(
                        y_pred,
                        index=y.index,
                    ).corr(
                        y,
                        method="pearson",
                    ),

                "spearman":
                    pd.Series(
                        y_pred,
                        index=y.index,
                    ).corr(
                        y,
                        method="spearman",
                    ),
            })


baseline_summary = pd.DataFrame(
    baseline_results
)


display(
    baseline_summary.round(4)
)

### 6.3 Linear models and incremental feature value

In [ ]:
# creating nested linear models and then evaluating on train and validation data
linear_feature_sets = {

    "Price_only": [
        "near_move_bps",
        "ref_move_bps",
    ],

    "Price_plus_auction": [
        "near_move_bps",
        "ref_move_bps",
        "signed_auction_imbalance_ratio",
        "side_sign",
    ],

    "Price_plus_auction_ADV": [
        "near_move_bps",
        "ref_move_bps",
        "signed_auction_imbalance_ratio",
        "side_sign",
        "signed_imbalance_adv",
    ],

    "Full_linear": [
        "near_move_bps",
        "ref_move_bps",
        "signed_auction_imbalance_ratio",
        "side_sign",
        "signed_imbalance_adv",
        "paired_adv",
        "spread_bps",
    ],
}


linear_results = []

for h in model_horizons:

    train_h = modeling_df[
        (
            modeling_df["split"] == "train"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    validation_h = modeling_df[
        (
            modeling_df["split"] == "validation"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    y_train = train_h[
        "cross_move_bps"
    ]

    y_validation = validation_h[
        "cross_move_bps"
    ]

    for model_name, features in (
        linear_feature_sets.items()
    ):

        linear_model = Pipeline([
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "regression",
                LinearRegression(),
            ),
        ])

        linear_model.fit(
            train_h[features],
            y_train,
        )

        for split_name, X, y in [
            (
                "train",
                train_h[features],
                y_train,
            ),
            (
                "validation",
                validation_h[features],
                y_validation,
            ),
        ]:

            prediction = linear_model.predict(
                X
            )

            prediction_series = pd.Series(
                prediction,
                index=y.index,
            )

            linear_results.append({

                "seconds_to_close":
                    h,

                "model":
                    model_name,

                "split":
                    split_name,

                "observations":
                    len(y),

                "mae_bps":
                    mean_absolute_error(
                        y,
                        prediction,
                    ),

                "rmse_bps":
                    np.sqrt(
                        mean_squared_error(
                            y,
                            prediction,
                        )
                    ),

                "pearson":
                    prediction_series.corr(
                        y,
                        method="pearson",
                    ),

                "spearman":
                    prediction_series.corr(
                        y,
                        method="spearman",
                    ),
            })


linear_summary = pd.DataFrame(
    linear_results
)

display(
    linear_summary.round(4)
)

In [ ]:
#checking the raw scale price only OLS coefficients
price_coefficient_results = []

price_features = [
    "near_move_bps",
    "ref_move_bps",
]

for h in model_horizons:

    train_h = modeling_df[
        (
            modeling_df["split"] == "train"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    validation_h = modeling_df[
        (
            modeling_df["split"] == "validation"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    price_model = LinearRegression()

    price_model.fit(
        train_h[price_features],
        train_h["cross_move_bps"],
    )

    validation_prediction = (
        price_model.predict(
            validation_h[price_features]
        )
    )

    price_coefficient_results.append({

        "seconds_to_close":
            h,

        "intercept_bps":
            price_model.intercept_,

        "near_coefficient":
            price_model.coef_[0],

        "ref_coefficient":
            price_model.coef_[1],

        "coefficient_sum":
            price_model.coef_.sum(),

        "validation_mae_bps":
            mean_absolute_error(
                validation_h["cross_move_bps"],
                validation_prediction,
            ),
    })


price_coefficient_summary = pd.DataFrame(
    price_coefficient_results
)

display(
    price_coefficient_summary.round(4)
)

In [ ]:
#ridge regularization and tune alpha using validation MAE only
ridge_feature_sets = {

    "Price_only": [
        "near_move_bps",
        "ref_move_bps",
    ],

    "Full_linear": [
        "near_move_bps",
        "ref_move_bps",
        "signed_auction_imbalance_ratio",
        "side_sign",
        "signed_imbalance_adv",
        "paired_adv",
        "spread_bps",
    ],
}


ridge_alphas = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]


ridge_results = []

for h in model_horizons:

    train_h = modeling_df[
        (
            modeling_df["split"] == "train"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    validation_h = modeling_df[
        (
            modeling_df["split"] == "validation"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    y_train = train_h[
        "cross_move_bps"
    ]

    y_validation = validation_h[
        "cross_move_bps"
    ]

    for feature_set_name, features in (
        ridge_feature_sets.items()
    ):

        for alpha in ridge_alphas:

            ridge_model = Pipeline([
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    ),
                ),
            ])

            ridge_model.fit(
                train_h[features],
                y_train,
            )

            validation_prediction = (
                ridge_model.predict(
                    validation_h[features]
                )
            )

            prediction_series = pd.Series(
                validation_prediction,
                index=y_validation.index,
            )

            ridge_results.append({

                "seconds_to_close":
                    h,

                "feature_set":
                    feature_set_name,

                "alpha":
                    alpha,

                "validation_mae_bps":
                    mean_absolute_error(
                        y_validation,
                        validation_prediction,
                    ),

                "validation_rmse_bps":
                    np.sqrt(
                        mean_squared_error(
                            y_validation,
                            validation_prediction,
                        )
                    ),

                "validation_pearson":
                    prediction_series.corr(
                        y_validation,
                        method="pearson",
                    ),

                "validation_spearman":
                    prediction_series.corr(
                        y_validation,
                        method="spearman",
                    ),
            })


ridge_summary = pd.DataFrame(
    ridge_results
)


best_ridge_by_horizon = (
    ridge_summary
    .sort_values(
        [
            "seconds_to_close",
            "feature_set",
            "validation_mae_bps",
        ]
    )
    .groupby(
        [
            "seconds_to_close",
            "feature_set",
        ],
        as_index=False,
    )
    .first()
    .sort_values(
        [
            "seconds_to_close",
            "feature_set",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


display(
    best_ridge_by_horizon.round(4)
)

### 6.4 Gradient boosted tree models

In [ ]:
#creating a conservative XGBoost benchmark to train on training dates and evaluate on validation dates
xgb_feature_sets = {

    "Price_XGB": [
        "near_move_bps",
        "ref_move_bps",
    ],

    "Full_XGB": [
        "near_move_bps",
        "ref_move_bps",
        "signed_auction_imbalance_ratio",
        "side_sign",
        "signed_imbalance_adv",
        "paired_adv",
        "spread_bps",
    ],
}


xgb_results = []

for h in model_horizons:

    train_h = modeling_df[
        (
            modeling_df["split"] == "train"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    validation_h = modeling_df[
        (
            modeling_df["split"] == "validation"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    y_train = train_h[
        "cross_move_bps"
    ]

    y_validation = validation_h[
        "cross_move_bps"
    ]

    for model_name, features in (
        xgb_feature_sets.items()
    ):

        xgb_model = XGBRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=10.0,
            reg_alpha=0.0,
            objective="reg:squarederror", #changed from absoluteerror to note changes
            random_state=42,
            n_jobs=-1,
        )

        xgb_model.fit(
            train_h[features],
            y_train,
        )

        for split_name, X, y in [
            (
                "train",
                train_h[features],
                y_train,
            ),
            (
                "validation",
                validation_h[features],
                y_validation,
            ),
        ]:

            prediction = xgb_model.predict(
                X
            )

            prediction_series = pd.Series(
                prediction,
                index=y.index,
            )

            xgb_results.append({

                "seconds_to_close":
                    h,

                "model":
                    model_name,

                "split":
                    split_name,

                "observations":
                    len(y),

                "mae_bps":
                    mean_absolute_error(
                        y,
                        prediction,
                    ),

                "rmse_bps":
                    np.sqrt(
                        mean_squared_error(
                            y,
                            prediction,
                        )
                    ),

                "pearson":
                    prediction_series.corr(
                        y,
                        method="pearson",
                    ),

                "spearman":
                    prediction_series.corr(
                        y,
                        method="spearman",
                    ),
            })


xgb_summary = pd.DataFrame(
    xgb_results
)

display(
    xgb_summary.round(4)
)

In [ ]:
# varying the max depths and min_child_weight on the full xgb model

xgb_depths = [
    2,
    3,
    4,
]

xgb_child_weights = [
    5,
    10,
    25,
]

full_xgb_features = [
    "near_move_bps",
    "ref_move_bps",
    "signed_auction_imbalance_ratio",
    "side_sign",
    "signed_imbalance_adv",
    "paired_adv",
    "spread_bps",
]


xgb_tuning_results = []

for h in model_horizons:

    train_h = modeling_df[
        (
            modeling_df["split"] == "train"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    validation_h = modeling_df[
        (
            modeling_df["split"] == "validation"
        )
        & (
            modeling_df["seconds_to_close"] == h
        )
    ].copy()

    X_train = train_h[
        full_xgb_features
    ]

    y_train = train_h[
        "cross_move_bps"
    ]

    X_validation = validation_h[
        full_xgb_features
    ]

    y_validation = validation_h[
        "cross_move_bps"
    ]

    for depth in xgb_depths:

        for child_weight in (
            xgb_child_weights
        ):

            model = XGBRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=depth,
                min_child_weight=child_weight,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=10.0,
                reg_alpha=0.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
            )

            model.fit(
                X_train,
                y_train,
            )

            train_prediction = (
                model.predict(
                    X_train
                )
            )

            validation_prediction = (
                model.predict(
                    X_validation
                )
            )

            xgb_tuning_results.append({

                "seconds_to_close":
                    h,

                "max_depth":
                    depth,

                "min_child_weight":
                    child_weight,

                "train_mae_bps":
                    mean_absolute_error(
                        y_train,
                        train_prediction,
                    ),

                "validation_mae_bps":
                    mean_absolute_error(
                        y_validation,
                        validation_prediction,
                    ),

                "validation_rmse_bps":
                    np.sqrt(
                        mean_squared_error(
                            y_validation,
                            validation_prediction,
                        )
                    ),
            })


xgb_tuning_summary = pd.DataFrame(
    xgb_tuning_results
)


best_xgb_by_horizon = (
    xgb_tuning_summary
    .sort_values(
        [
            "seconds_to_close",
            "validation_mae_bps",
        ]
    )
    .groupby(
        "seconds_to_close",
        as_index=False,
    )
    .first()
    .sort_values(
        "seconds_to_close",
        ascending=False,
    )
)


display(
    best_xgb_by_horizon.round(4)
)

In [ ]:
#creating a pooled multi horizon XGBoost to compare with the previous results
pooled_xgb_features = [
    "seconds_to_close",
    "near_move_bps",
    "ref_move_bps",
    "signed_auction_imbalance_ratio",
    "side_sign",
    "signed_imbalance_adv",
    "paired_adv",
    "spread_bps",
]


pooled_train = modeling_df[
    modeling_df["split"] == "train"
].copy()

pooled_validation = modeling_df[
    modeling_df["split"] == "validation"
].copy()


pooled_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10.0,
    reg_alpha=0.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)


pooled_xgb.fit(
    pooled_train[pooled_xgb_features],
    pooled_train["cross_move_bps"],
)


pooled_validation[
    "pooled_xgb_prediction"
] = pooled_xgb.predict(
    pooled_validation[
        pooled_xgb_features
    ]
)


pooled_results = []

for h in model_horizons:

    g = pooled_validation[
        pooled_validation[
            "seconds_to_close"
        ] == h
    ].copy()

    y = g[
        "cross_move_bps"
    ]

    prediction = g[
        "pooled_xgb_prediction"
    ]

    pooled_results.append({

        "seconds_to_close":
            h,

        "observations":
            len(g),

        "validation_mae_bps":
            mean_absolute_error(
                y,
                prediction,
            ),

        "validation_rmse_bps":
            np.sqrt(
                mean_squared_error(
                    y,
                    prediction,
                )
            ),

        "validation_pearson":
            prediction.corr(
                y,
                method="pearson",
            ),

        "validation_spearman":
            prediction.corr(
                y,
                method="spearman",
            ),
    })


pooled_xgb_summary = pd.DataFrame(
    pooled_results
)

display(
    pooled_xgb_summary.round(4)
)

### 6.5 Model selection and final out-of-sample evaluation

Validation MAE is the primary model-selection metric. When two specifications perform similarly, I prefer the simpler model.

Model choices are made using the training and validation periods only. The final test period is held back until feature selection, model architecture, and hyperparameter choices are fixed.

The public notebook preserves the model-selection workflow and final-evaluation code, while execution outputs from the original dataset are intentionally removed.


In [ ]:
#creating the final test evaluation
development_df = modeling_df[
    modeling_df["split"].isin(
        [
            "train",
            "validation",
        ]
    )
].copy()

test_df = modeling_df[
    modeling_df["split"] == "test"
].copy()

#freezing the pooled XGBoost feature set

final_pooled_features = [
    "seconds_to_close",
    "near_move_bps",
    "ref_move_bps",
    "signed_auction_imbalance_ratio",
    "side_sign",
    "signed_imbalance_adv",
    "paired_adv",
    "spread_bps",
]

#fitting the final pooled XGBoost on all horizons thereby preserving the validated architecture
final_pooled_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10.0,
    reg_alpha=0.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

final_pooled_xgb.fit(
    development_df[
        final_pooled_features
    ],
    development_df[
        "cross_move_bps"
    ],
)


test_df[
    "pooled_xgb_prediction"
] = final_pooled_xgb.predict(
    test_df[
        final_pooled_features
    ]
)

#evaluating the frozen horizon architecture

final_results = []

for h in model_horizons:

    development_h = development_df[
        development_df[
            "seconds_to_close"
        ] == h
    ].copy()

    test_h = test_df[
        test_df[
            "seconds_to_close"
        ] == h
    ].copy()

    y_development = development_h[
        "cross_move_bps"
    ]

    y_test = test_h[
        "cross_move_bps"
    ]


    if h in [
        120,
        60,
        30,
    ]:

        selected_model = (
            "Pooled_Full_XGB"
        )

        test_prediction = (
            test_h[
                "pooled_xgb_prediction"
            ]
            .to_numpy()
        )


    elif h in [
        10,
        5,
    ]:

        selected_model = (
            "Price_OLS"
        )

        price_features = [
            "near_move_bps",
            "ref_move_bps",
        ]

        final_price_model = (
            LinearRegression()
        )

        final_price_model.fit(
            development_h[
                price_features
            ],
            y_development,
        )

        test_prediction = (
            final_price_model.predict(
                test_h[
                    price_features
                ]
            )
        )


    elif h == 1:

        selected_model = "Near"

        test_prediction = (
            test_h[
                "near_move_bps"
            ]
            .to_numpy()
        )


    #selected model evaluation

    prediction_series = pd.Series(
        test_prediction,
        index=y_test.index,
    )

    selected_mae = float(
        mean_absolute_error(
            y_test,
            test_prediction,
        )
    )

    selected_rmse = float(
        np.sqrt(
            mean_squared_error(
                y_test,
                test_prediction,
            )
        )
    )

    selected_pearson = float(
        prediction_series.corr(
            y_test,
            method="pearson",
        )
    )

    selected_spearman = float(
        prediction_series.corr(
            y_test,
            method="spearman",
        )
    )


    #simple market baselines
    midpoint_prediction = np.zeros(
        len(test_h)
    )

    reference_prediction = (
        test_h[
            "ref_move_bps"
        ]
        .to_numpy()
    )

    near_prediction = (
        test_h[
            "near_move_bps"
        ]
        .to_numpy()
    )


    midpoint_mae = float(
        mean_absolute_error(
            y_test,
            midpoint_prediction,
        )
    )

    reference_mae = float(
        mean_absolute_error(
            y_test,
            reference_prediction,
        )
    )

    near_mae = float(
        mean_absolute_error(
            y_test,
            near_prediction,
        )
    )

    baseline_maes = [
        (
            "Midpoint",
            midpoint_mae,
        ),
        (
            "Reference",
            reference_mae,
        ),
        (
            "Near",
            near_mae,
        ),
    ]

    (
        best_market_baseline,
        best_market_baseline_mae,
    ) = min(
        baseline_maes,
        key=lambda item: item[1],
    )


    final_results.append({

        "seconds_to_close":
            h,

        "selected_model":
            selected_model,

        "observations":
            len(y_test),

        "test_mae_bps":
            selected_mae,

        "test_rmse_bps":
            selected_rmse,

        "test_pearson":
            selected_pearson,

        "test_spearman":
            selected_spearman,

        "midpoint_mae_bps":
            midpoint_mae,

        "reference_mae_bps":
            reference_mae,

        "near_mae_bps":
            near_mae,

        "best_market_baseline":
            best_market_baseline,

        "best_market_baseline_mae_bps":
            best_market_baseline_mae,

        "mae_improvement_vs_best_baseline_bps":
            (
                best_market_baseline_mae
                - selected_mae
            ),
    })


final_test_summary = pd.DataFrame(
    final_results
)

display(
    final_test_summary.round(4)
)

The public portfolio version intentionally omits dataset-specific held-out performance values.

The important methodological point is that the final test set is opened only after the validation-driven model choices are frozen. The notebook then compares the selected model against contemporaneous price baselines at each prediction horizon.


### 6.6 Inspecting the test results

In [ ]:
# checking the test period robustness by trading date
test_diagnostic_df = test_df.copy()

test_diagnostic_df[
    "selected_prediction"
] = np.nan

test_diagnostic_df[
    "selected_model"
] = ""

#pooled xgboost predictions
early_mask = (
    test_diagnostic_df[
        "seconds_to_close"
    ].isin(
        [
            120,
            60,
            30,
        ]
    )
)

test_diagnostic_df.loc[
    early_mask,
    "selected_prediction"
] = test_diagnostic_df.loc[
    early_mask,
    "pooled_xgb_prediction"
]

test_diagnostic_df.loc[
    early_mask,
    "selected_model"
] = "Pooled_Full_XGB"


#price ols predictions
price_features = [
    "near_move_bps",
    "ref_move_bps",
]

for h in [
    10,
    5,
]:

    development_h = development_df[
        development_df[
            "seconds_to_close"
        ] == h
    ].copy()

    test_mask = (
        test_diagnostic_df[
            "seconds_to_close"
        ] == h
    )

    final_price_model = LinearRegression()

    final_price_model.fit(
        development_h[
            price_features
        ],
        development_h[
            "cross_move_bps"
        ],
    )

    test_diagnostic_df.loc[
        test_mask,
        "selected_prediction"
    ] = final_price_model.predict(
        test_diagnostic_df.loc[
            test_mask,
            price_features,
        ]
    )

    test_diagnostic_df.loc[
        test_mask,
        "selected_model"
    ] = "Price_OLS"

#near price prediction
one_second_mask = (
    test_diagnostic_df[
        "seconds_to_close"
    ] == 1
)

test_diagnostic_df.loc[
    one_second_mask,
    "selected_prediction"
] = test_diagnostic_df.loc[
    one_second_mask,
    "near_move_bps"
]

test_diagnostic_df.loc[
    one_second_mask,
    "selected_model"
] = "Near"


#evaluating each test date and horizon separately
daily_results = []

for (
    test_date,
    h,
), group in test_diagnostic_df.groupby(
    [
        "date",
        "seconds_to_close",
    ]
):

    y = group[
        "cross_move_bps"
    ]

    selected_prediction = group[
        "selected_prediction"
    ]

    midpoint_prediction = np.zeros(
        len(group)
    )

    reference_prediction = group[
        "ref_move_bps"
    ]

    near_prediction = group[
        "near_move_bps"
    ]


    selected_mae = float(
        mean_absolute_error(
            y,
            selected_prediction,
        )
    )

    midpoint_mae = float(
        mean_absolute_error(
            y,
            midpoint_prediction,
        )
    )

    reference_mae = float(
        mean_absolute_error(
            y,
            reference_prediction,
        )
    )

    near_mae = float(
        mean_absolute_error(
            y,
            near_prediction,
        )
    )


    baseline_maes = [
        (
            "Midpoint",
            midpoint_mae,
        ),
        (
            "Reference",
            reference_mae,
        ),
        (
            "Near",
            near_mae,
        ),
    ]

    (
        best_market_baseline,
        best_market_baseline_mae,
    ) = min(
        baseline_maes,
        key=lambda item: item[1],
    )


    daily_results.append({

        "date":
            test_date,

        "seconds_to_close":
            h,

        "selected_model":
            group[
                "selected_model"
            ].iloc[0],

        "observations":
            len(group),

        "selected_mae_bps":
            selected_mae,

        "midpoint_mae_bps":
            midpoint_mae,

        "reference_mae_bps":
            reference_mae,

        "near_mae_bps":
            near_mae,

        "best_market_baseline":
            best_market_baseline,

        "best_market_baseline_mae_bps":
            best_market_baseline_mae,

        "mae_improvement_vs_best_baseline_bps":
            (
                best_market_baseline_mae
                - selected_mae
            ),

        "beats_best_baseline":
            (
                selected_mae
                < best_market_baseline_mae
            ),
    })


daily_test_summary = (
    pd.DataFrame(
        daily_results
    )
    .sort_values(
        [
            "date",
            "seconds_to_close",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    daily_test_summary.round(4)
)

I also evaluate the frozen forecasts separately by held-out trading date rather than relying only on one aggregate test metric.

This is a robustness check for temporal concentration: a model that looks good in aggregate should not owe all of its improvement to a single favorable date. The public version keeps this diagnostic logic but clears the original execution output.


In [ ]:
#creating error distribution and tail risk

error_diagnostic_df = (
    test_diagnostic_df.copy()
)


error_diagnostic_df[
    "forecast_error_bps"
] = (
    error_diagnostic_df[
        "selected_prediction"
    ]
    - error_diagnostic_df[
        "cross_move_bps"
    ]
)

error_diagnostic_df[
    "absolute_error_bps"
] = (
    error_diagnostic_df[
        "forecast_error_bps"
    ]
    .abs()
)

error_distribution_results = []

for h in model_horizons:

    group = error_diagnostic_df[
        error_diagnostic_df[
            "seconds_to_close"
        ] == h
    ].copy()

    absolute_error = group[
        "absolute_error_bps"
    ]

    signed_error = group[
        "forecast_error_bps"
    ]


    error_distribution_results.append({

        "seconds_to_close":
            h,

        "selected_model":
            group[
                "selected_model"
            ].iloc[0],

        "observations":
            len(group),

        "mean_signed_error_bps":
            float(
                signed_error.mean()
            ),

        "mae_bps":
            float(
                absolute_error.mean()
            ),

        "median_abs_error_bps":
            float(
                absolute_error.quantile(
                    0.50
                )
            ),

        "p90_abs_error_bps":
            float(
                absolute_error.quantile(
                    0.90
                )
            ),

        "p95_abs_error_bps":
            float(
                absolute_error.quantile(
                    0.95
                )
            ),

        "p99_abs_error_bps":
            float(
                absolute_error.quantile(
                    0.99
                )
            ),

        "max_abs_error_bps":
            float(
                absolute_error.max()
            ),

        "share_error_gt_10bps":
            float(
                (
                    absolute_error > 10
                ).mean()
            ),

        "share_error_gt_25bps":
            float(
                (
                    absolute_error > 25
                ).mean()
            ),
    })


error_distribution_summary = (
    pd.DataFrame(
        error_distribution_results
    )
    .sort_values(
        "seconds_to_close",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    error_distribution_summary.round(4)
)

In [ ]:
#inspecting the largest held out forecast errors

tail_error_cases = (
    error_diagnostic_df
    .sort_values(
        [
            "seconds_to_close",
            "absolute_error_bps",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .groupby(
        "seconds_to_close",
        group_keys=False,
    )
    .head(3)
    [
        [
            "date",
            "symbol",
            "seconds_to_close",
            "selected_model",
            "cross_move_bps",
            "selected_prediction",
            "forecast_error_bps",
            "absolute_error_bps",
            "near_move_bps",
            "ref_move_bps",
            "signed_auction_imbalance_ratio",
            "signed_imbalance_adv",
            "paired_adv",
            "spread_bps",
        ]
    ]
    .sort_values(
        [
            "seconds_to_close",
            "absolute_error_bps",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

display(
    tail_error_cases.round(4)
)

## 7. Conclusions and limitations

The project is designed around a simple question: how does the information set for predicting the closing cross change as the auction gets closer?

The research workflow compares simple market-price baselines with progressively richer models using indicative auction prices, imbalance, volume, and liquidity features. It uses fixed prediction horizons and a chronological train/validation/test design to avoid mixing observations from different stages of the auction or leaking information across time.

A key modeling principle is that complexity has to earn its place. Linear models, regularized models, and gradient-boosted trees are compared against simple price baselines, and the final model is chosen by held-out validation rather than by in-sample fit.

The main limitation of any short-window auction study is regime coverage. A stronger production study would extend the analysis across a much longer history, additional volatility regimes, and a broader set of market conditions. It would also separate forecasting accuracy from actual trading profitability by incorporating execution costs, auction participation mechanics, market impact, and position constraints.

The original source dataset and dataset-specific outputs are intentionally not redistributed in this public portfolio version.
